# Fine-tuning Sentiment Analysis

In this notebook we take a pre-trained distilBERT model (a small and fast transformer model trained by distilling BERT base) that has an un-trained head for sentiment analysis. In other words, the head needs to be trained on given examples so that the model learns to catagorize sequences of text into either 'positive' or 'negative' categories.

>This model is a distilled version of the [BERT base model](https://huggingface.co/bert-base-uncased). It was introduced in this [paper](https://arxiv.org/abs/1910.01108). The code for the distillation process can be found here. This model is uncased: it does not make a difference between english and English.

The model is trained with the following dataset (loaded from Huggingface)

* [GLUE](https://gluebenchmark.com/) (General Language Understanding Evaluation)

In [1]:
import transformers
import torch
import evaluate
from transformers import AutoTokenizer
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
from transformers import Trainer
from transformers import pipeline
from datasets import load_dataset
#from datasets import load_metric
import numpy as np
from torchinfo import summary
from typing import Tuple, Dict
import json
from pprint import pp

In [2]:
print(f"Torch GPU acceleration available: {torch.cuda.is_available()}")
print(f"Torch CUDA version: {torch.version.cuda}")
print(f"Torch supported architecture: {torch.cuda.get_arch_list()}")

Torch GPU acceleration available: True
Torch CUDA version: 12.8
Torch supported architecture: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


In [3]:
# GLUE stands for General Language Understanding Evaluation
# Loads sst2 (sentiment analysis) task data from the GLUE dataset
raw_dataset = load_dataset("glue", "sst2")

In [4]:
# Show contents of the dataset. In essence we have the following:
# - training datast
# - validation dataset
# - test dataset
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [5]:
# The model to be used and the corresponding tokenizer
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
# Let's tokenize a few samples
tokenized_samples = tokenizer(raw_dataset['train'][0:3]['sentence'])

In [7]:
# Print the tokenized samples
pp(tokenized_samples)

{'input_ids': [[101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102],
               [101,
                3397,
                2053,
                15966,
                1010,
                2069,
                4450,
                2098,
                18201,
                2015,
                102],
               [101,
                2008,
                7459,
                2049,
                3494,
                1998,
                10639,
                2015,
                2242,
                2738,
                3376,
                2055,
                2529,
                3267,
                102]],
 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
                    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


In [8]:
# Print a few samples
print(f"phrases: {raw_dataset['train'][0:3]['sentence']}:\ntoken ids:{tokenized_samples}")

phrases: ['hide new secretions from the parental units ', 'contains no wit , only labored gags ', 'that loves its characters and communicates something rather beautiful about human nature ']:
token ids:{'input_ids': [[101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], [101, 3397, 2053, 15966, 1010, 2069, 4450, 2098, 18201, 2015, 102], [101, 2008, 7459, 2049, 3494, 1998, 10639, 2015, 2242, 2738, 3376, 2055, 2529, 3267, 102]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


In [9]:
# Tokenizer function to apply tokenization to all of the datasets
def tokenizer_fn(batch):
    return tokenizer(batch['sentence'], truncation=True)

In [10]:
tokenized_datasets = raw_dataset.map(tokenizer_fn, batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [11]:
print(type(tokenized_datasets))

<class 'datasets.dataset_dict.DatasetDict'>


In [12]:
# Training parameters. Train only for 1 epoch.
training_parameters = TrainingArguments(
    'distil-bert-trainer',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=1,
)

In [13]:
# Create the model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
# Print out the type of the model
type(model)

transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification

In [15]:
# Show the model layers
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [16]:
# Print summary of the model
summary(model)

Layer (type:depth-idx)                                  Param #
DistilBertForSequenceClassification                     --
├─DistilBertModel: 1-1                                  --
│    └─Embeddings: 2-1                                  --
│    │    └─Embedding: 3-1                              23,440,896
│    │    └─Embedding: 3-2                              393,216
│    │    └─LayerNorm: 3-3                              1,536
│    │    └─Dropout: 3-4                                --
│    └─Transformer: 2-2                                 --
│    │    └─ModuleList: 3-5                             42,527,232
├─Linear: 1-2                                           590,592
├─Linear: 1-3                                           1,538
├─Dropout: 1-4                                          --
Total params: 66,955,010
Trainable params: 66,955,010
Non-trainable params: 0

In [17]:
# Store the weights to a list so that we can compare the weights
params_before_training = []
for name, p in model.named_parameters():
    params_before_training.append(p.detach().cpu().numpy())

In [18]:
# Create a metric object for evaluating the performance
metric = evaluate.load("glue", "sst2")

In [19]:
# Show how the metric object works
metric.compute(predictions=[1,0,1], references=[1,0,0])

{'accuracy': 0.6666666666666666}

In [20]:
# Create a compute_metrics function that is passed to the Trainer-object
def compute_metrics(logits_and_labels: Tuple) -> Dict:
    """Computes accuracy metrics based on the given logits from the network and the correct labels.

    Parameters
    ----------
    logits_and_labels: Tuple
        Contains the logits from the network and the actual labels

    Return
    ------
    Dict
        A dictionary in the format {'accuracy': val}
    """
    
    logits, labels = logits_and_labels
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [21]:
# Trainer object used for training the model
trainer = Trainer(
    model,
    training_parameters,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_818/3026859003.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.197300,0.362077,0.903670


TrainOutput(global_step=8419, training_loss=0.2634601713090601, metrics={'train_runtime': 127.9919, 'train_samples_per_second': 526.197, 'train_steps_per_second': 65.778, 'total_flos': 517212489917652.0, 'train_loss': 0.2634601713090601, 'epoch': 1.0})

In [23]:
trainer.save_model("trained_distilbert_model")

In [24]:
if torch.cuda.is_available():
    print("Model is execute in: GPU")
    sequence_classifier = pipeline("text-classification", model="trained_distilbert_model", device=torch.cuda.current_device())
else:
    print("Model is execute in: CPU")
    sequence_classifier = pipeline("text-classification", model="trained_distilbert_model")

Device set to use cuda:0


Model is execute in: GPU


In [25]:
# Sentiment label to text
label2sentiment={"LABEL_0": "negative", "LABEL_1": "positive"}

In [26]:
# Test the new model
phrase = "This movie sucks big time!"
sentiment = sequence_classifier(phrase)
print(f"{phrase} -> sentiment: {label2sentiment[sentiment[0]['label']]}")

This movie sucks big time! -> sentiment: negative


In [27]:
# Test the new model
phrase = "This movie is fantastic!"
sentiment = sequence_classifier(phrase)
print(f"{phrase} -> sentiment: {label2sentiment[sentiment[0]['label']]}")

This movie is fantastic! -> sentiment: positive


In [28]:
# Show the model configuration
model_config_file="trained_distilbert_model/config.json"

In [29]:
# Read in the config-file of the trained model
with open(model_config_file) as f:
    model_config = json.load(f)

In [30]:
# Show the model parameters
for key in model_config:
    print(f"{key}: {model_config[key]}")

activation: gelu
architectures: ['DistilBertForSequenceClassification']
attention_dropout: 0.1
dim: 768
dropout: 0.1
dtype: float32
hidden_dim: 3072
initializer_range: 0.02
max_position_embeddings: 512
model_type: distilbert
n_heads: 12
n_layers: 6
pad_token_id: 0
problem_type: single_label_classification
qa_dropout: 0.1
seq_classif_dropout: 0.2
sinusoidal_pos_embds: False
tie_weights_: True
transformers_version: 4.57.1
vocab_size: 30522
